In [ ]:
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = Path(r"...")
DATA_FILE = DATA_DIR / "main_lentele_50.parquet"

main = pd.read_parquet(DATA_FILE)
main["DATE"] = pd.to_datetime(main["DATE"])

Vakar arba ryt bus šventė

`HOLIDAY_FLAG` požymis sudarytas pagal iš anksto apibrėžtą Lietuvos šventinių dienų sąrašą 2023–2025 m. laikotarpiui. Kadangi nagrinėjamos parduotuvės Velykų sekmadienį ir pirmąją Kalėdų dieną nedirba, šios dienos tiesiogiai nežymimos kaip šventinės pardavimų dienos. Vietoje Velykų sekmadienio žymima diena prieš Velykas, nes ji gali turėti reikšmingą poveikį pirkėjų elgsenai.

Į `HOLIDAY_FLAG` įtraukiamos šios pasikartojančios kalendorinės dienos:

- Naujieji metai;
- Lietuvos valstybės atkūrimo diena;
- Lietuvos nepriklausomybės atkūrimo diena;
- diena prieš Velykas;
- antroji Velykų diena;
- Tarptautinė darbo diena;
- Motinos diena;
- Tėvo diena;
- Joninės;
- Valstybės diena;
- Žolinė;
- Visų Šventųjų diena;
- Vėlinės;
- Kūčios;
- antroji Kalėdų diena.

In [2]:
calendar = (
    main[["DATE", "HOLIDAY_FLAG"]]
    .drop_duplicates()
    .sort_values("DATE")
    .copy()
)

holiday_lookup = calendar.set_index("DATE")["HOLIDAY_FLAG"]

main["YESTERDAY_HOLIDAY_FLAG"] = (
    main["DATE"].sub(pd.Timedelta(days=1))
    .map(holiday_lookup)
    .fillna(0)
    .astype(int)
)

main["TOMORROW_HOLIDAY_FLAG"] = (
    main["DATE"].add(pd.Timedelta(days=1))
    .map(holiday_lookup)
    .fillna(0)
    .astype(int)
)

Kokia savaitės diena?

In [3]:
main["DAY_OF_WEEK"] = main["DATE"].dt.dayofweek  # Monday=0, Sunday=6

Penktadienis, šeštadienis ir sekmdadienis = savaitgalis_flag

In [4]:
main["WEEKEND_FLAG"] = main["DAY_OF_WEEK"].isin([4, 5, 6]).astype(int)

In [ ]:
OUTPUT_FILE = DATA_DIR / "main_lentele_50_su_features.parquet"

main.to_parquet(
    OUTPUT_FILE,
    index=False
)